# Real-Estate Price Regression with `sltn`

This notebook demonstrates how to formulate a regression task in `sltn` using a learnable function and a differentiable equality predicate.

## Problem

The goal is to predict the house price per unit area from six real-estate features. We represent the predictor as a learnable function:

$$ f : Sample \to Price. $$

Regression is expressed through a smooth equality predicate:

$$
Eq(u, v) = \exp \left(-\alpha \, \|u - v\|\right),
$$

where $\alpha > 0$ controls how quickly satisfaction decreases as the prediction error grows.

The knowledge base contains one axiom:

$$
\mathcal{K}
=
\left\{
\forall (x, y) \; Eq(f(x), y)
\right\}.
$$

Each feature vector $x_i$ is paired with its corresponding target price $y_i$.

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Dataset: Real Estate

The dataset contains real-estate transactions represented by six numerical features and one target value: the house price per unit area.
The following cell loads the local CSV file, shuffles the rows, extracts the six input features and the target price, and creates training and test splits.

In [ ]:
import pandas as pd

data = pd.read_csv("../datasets/real-estate.csv")
data = data.sample(frac=1)   # shuffle

feature_cols = [
    'X1 transaction date', 'X2 house age',
    'X3 distance to the nearest MRT station',
    'X4 number of convenience stores',
    'X5 latitude', 'X6 longitude',
]
x = torch.tensor(data[feature_cols].to_numpy()).float()
y = torch.tensor(data[['Y house price of unit area']].to_numpy()).float()

x_train, y_train = x[:330], y[:330]
x_test,  y_test  = x[330:], y[330:]

## 1. Signature

The signature declares the symbolic vocabulary for the regression problem.

It contains:

- the sort `Sample`, representing a six-feature real-estate sample;
- the sort `Price`, representing a scalar target value;
- the function `f : Sample → Price`, which will be grounded by a neural regressor;
- the predicate `Eq : Price × Price → Bool`, used to compare predicted and observed prices;
- variables `x` and `y` for samples and target prices.

The signature is symbolic only; concrete tensors and neural modules are assigned in the interpretation.

In [ ]:
import sltn
from sltn import Signature, Interpretation

sig    = Signature("regression")
sig.sort("Sample")  # 6-feature real-estate sample
sig.sort("Price")   # house price (scalar)
sig.function("f", ["Sample"], "Price")   # learnable regression function
sig.predicate("Eq", ["Price", "Price"])  # smooth equality predicate

sig.variable("x", "Sample")
sig.variable("y", "Price")

print(sig)

## 2. Formula

The regression objective is expressed as a single universally quantified formula:

$$
\forall (x, y) \; Eq(f(x), y).
$$

This states that every predicted price $f(x)$ should be close to the corresponding observed price $y$. In the data loader, `x` and `y` are assigned batch tensors with matching order, so each row of features is evaluated against its own target value.

In [ ]:
# ∀(x, y) Eq(f(x), y)   — diagonal quantifier pairs each x_i with y_i
phi_eq = sig.parse("forall (x, y): Eq(f(x), y)")

print("phi_eq:", phi_eq)

## 3. Neural model

The function `f` is grounded by a small multilayer perceptron. The model receives a six-dimensional input vector and returns one scalar prediction. The output layer is linear.

In [ ]:
class RegressionMLP(torch.nn.Module):
    """Predicts house price from 6 features.  Output is linear (no activation)."""
    def __init__(self, layer_sizes=(6, 8, 8, 1)):
        super().__init__()
        self.elu = torch.nn.ELU()
        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(layer_sizes[i-1], layer_sizes[i])
            for i in range(1, len(layer_sizes))
        ])

    def forward(self, x):
        for layer in self.layers[:-1]: x = self.elu(layer(x))
        return self.layers[-1](x)

## 4. Interpretation

The interpretation grounds the symbols declared in the signature.

In this notebook:

- `Sample` is interpreted as a six-dimensional tensor type;
- `Price` is interpreted as a scalar tensor type;
- `f` is interpreted as the regression MLP;
- `Eq` is interpreted as a smooth distance-based equality predicate.

The predicate returns values in $(0, 1]$: predictions close to the target price receive high satisfaction, while larger errors receive lower satisfaction.

In [ ]:
interp = Interpretation(sig).to(device)

interp["Sample"] = sltn.Type(name = "Sample", shape= (x_train.shape[1],))
interp["Price"]  = sltn.Type(name = "Price", shape= (y_train.shape[1],))

interp["f"] = RegressionMLP()

alpha = 0.05
interp["Eq"]= lambda u, v: torch.exp(-alpha * torch.sqrt(torch.sum(torch.square(u - v), dim=-1, keepdim=True)))

print(interp)

## 5. Data loader and metrics

This section defines a data loader and two evaluation metrics. The satisfaction level is obtained by evaluating the regression formula on each batch. The RMSE metric is computed separately as a standard regression measure, to interpret numerical prediction quality alongside logical satisfaction.

In [ ]:
from sklearn.metrics import mean_squared_error

class RawDataLoader:
    def __init__(self, x, y, batch_size=64, shuffle=True):
        self.x = x.to(device); self.y = y.to(device)
        self.batch_size = batch_size; self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.x.shape[0] / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.x.shape[0]))
        if self.shuffle: np.random.shuffle(idxlist)
        for start in range(0, self.x.shape[0], self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.x[idx], self.y[idx]

train_loader = RawDataLoader(x_train, y_train, batch_size=64)
test_loader  = RawDataLoader(x_test,  y_test,  batch_size=64, shuffle=False)

def compute_sat_level(loader):
    mean_sat = 0
    for x, y in loader:
        interp["x"] = x
        interp["y"] = y
        mean_sat += interp(phi_eq).value
    return mean_sat / len(loader)

def compute_rmse(loader):
    mean_rmse = 0.0
    for x, y in loader:
        interp["x"] = x
        interp["y"] = y
        preds = interp["f"](x).detach().cpu().numpy()
        mean_rmse += mean_squared_error(y.cpu().numpy(), preds)
    return mean_rmse / len(loader)

## 6. Training loop

Training maximizes the satisfaction of the regression formula.

For each batch:

1. feature vectors are assigned to `x`;
2. target prices are assigned to `y`;
3. the formula is evaluated under the current interpretation;
4. the loss is defined as: $\mathcal{L} = 1 - \mathrm{SatAgg}(\{\forall (x, y) \; Eq(f(x), y)\});$
5. gradients are backpropagated through the equality predicate and the neural regressor;
6. the optimizer updates the parameters of `f`.

The printed metrics report the training loss, satisfaction levels, and RMSE on both training and test sets.

In [ ]:
optimizer = torch.optim.Adam(interp["f"].parameters(), lr=0.0005)

for epoch in range(500):
    train_loss = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        interp["x"] = x
        interp["y"] = y
        loss = 1 - interp(phi_eq).value
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch % 50 == 0:
        print(
            f"epoch {epoch:3d} | loss {train_loss:.4f} "
            f"| Train Sat {compute_sat_level(train_loader).item():.3f} "
            f"| Test Sat {compute_sat_level(test_loader).item():.3f} "
            f"| Train RMSE {compute_rmse(train_loader):.3f} "
            f"| Test RMSE {compute_rmse(test_loader):.3f}"
        )